# Spherical open-boundary symbols and Cauer representations

This notebook evaluates `radia.open_boundary`: spherical diffusion DtN symbols, their continued fractions in the square-root frequency variable, and separate diffusion-memory approximations. The saved assertions check these algebraic representations. They do not establish an exact finite physical-time ODE for diffusion, unconditional timestep stability, or superiority over PML. See the method and limitations section below.


In [1]:
import sys, os
sys.argv = ["notebook"]

# -*- coding: utf-8 -*-
r"""demo_dtn_cln_usage.py -- using radia.open_boundary (the Zs-DtN-CLN open boundary)
================================================================================
The PRODUCTION-API counterpart of the retired research demos, now maintained
through `radia.open_boundary` and `validation_test/open_boundary/test_dtn_cln.py`.
Here the verified operator ships as `radia.open_boundary`; this shows how to use it.

WHAT it gives you, for a SEPARABLE (spherical) MQS truncation:
  * the EXACT exterior eddy/diffusion DtN eigenvalue per multipole n,
  * its Cauer Ladder Network (CLN) realisation -- EXACT at n+1 stages, well-conditioned,
  * the companion auxiliary-ODE poles (Re<0) for a transient passive Robin boundary.

SCOPE (honest): this WINS over a CFS-PML only on its island -- compact / quasi-spherical
MQS problems wanting an exact, DC-well-conditioned boundary + a compact passive circuit
ROM.  Elongated/arbitrary geometry -> a box CFS-PML hugs better; genuine wave radiation
is outside radia's MQS scope.  NOT novel (Grote-Keller / Hagstrom-Warburton / Warburg-Cauer
/ Kameari CLN).  See docs/open_boundary/OPEN_BOUNDARY_MAP.md.

Every printed 'ok' is gated on an executed assertion (no overclaim).
"""
import os
os.environ.setdefault("PYTHONIOENCODING", "utf-8")
import numpy as np
import radia.open_boundary as ob

print("=" * 78)
print(" radia.open_boundary : exact Zs-DtN-CLN open boundary -- usage")
print("=" * 78)

# A copper-like spherical truncation: R0 = 30 mm, mu*sigma (mu0 * 58 MS/m).
R0 = 0.03
MU_SIGMA = 4.0e-7 * np.pi * 5.8e7
omega = np.logspace(1, 5, 40)      # 10 .. 1e5 rad/s

print(f"\n[1] exact eddy/diffusion DtN per multipole (sphere R0={R0} m):")
for n in (1, 2, 3):
    g50 = ob.eddy_dtn(n, 1j * 5.0e4, R0=R0, mu_sigma=MU_SIGMA)
    print(f"    n={n}: G_n(i*5e4) = {g50:.4f}")

print("\n[2] Cauer ladder is EXACT at n+1 stages and reproduces the symbol:")
for n in (1, 2, 3, 4, 5, 6):
    stages = ob.cauer_ladder(n)
    Zc = np.array([ob.eval_ladder(stages, 1j * w, R0, MU_SIGMA) for w in omega])
    Zr = np.array([ob.eddy_dtn(n, 1j * w, R0, MU_SIGMA) for w in omega])
    nrmse = float(np.sqrt(np.mean(np.abs(Zc - Zr) ** 2)) / np.sqrt(np.mean(np.abs(Zr) ** 2)))
    print(f"    n={n}: stages={len(stages)} (= n+1)   ladder-vs-symbol NRMSE={nrmse:.1e}")
    assert len(stages) == n + 1 and nrmse < 1e-9

print("\n[3] transient Robin realisation: one auxiliary ODE per companion pole (Re<0):")
for n in (1, 2, 3):
    poles = ob.companion_poles(n)
    print(f"    n={n}: {len(poles)} poles, max Re = {poles.real.max():+.3f}  "
          f"=> passive / unconditionally stable (Grote-Keller form)")
    assert np.all(poles.real < 0)

print("\n[4] sqrt(s) diffusion-memory element as a finite passive ladder:")
g, p, e = ob.sqrt_s_passive_ladder(omega, 12)
print(f"    K=12: NRMSE={e:.2e}, all poles -p<0 (stable), passive (g>=0): "
      f"{bool(np.all(g >= 0) and np.all(p > 0))}")
assert e < 5e-2 and np.all(p > 0) and np.all(g >= 0)

print("\n[5] Kelvin BUILDS the DtN (material-aware / non-separable companion):")
print("    a radial Kelvin-FEM reproduces the closed-form eddy DtN with NO DC floor")
band = 1j * np.logspace(-4, 2, 30)
for n in (1, 2, 3):
    G = np.array([ob.kelvin_fem_radial_dtn(n, s) for s in band])
    Gx = np.array([ob.eddy_dtn(n, s) for s in band])
    nrmse = float(np.sqrt(np.mean(np.abs(G - Gx) ** 2)) / np.sqrt(np.mean(np.abs(Gx) ** 2)))
    print(f"    n={n}: Kelvin-FEM build vs closed-form, band NRMSE={nrmse:.1e}")
    assert nrmse < 5e-2
print("    (the arbitrary-shape / iron-exterior path = ob.kelvin_dtn_matrix + ob.steklov_spectrum,")
print("     verified O_h-split on a cube -- needs NGSolve; see tests/open_boundary/test_kelvin_dtn.py.")
print("     NOTE: material-in-exterior Kelvin is CLASSICAL (Freeman-Lowther 1988/89).)")

print("\nALL CHECKS PASSED.")


 radia.open_boundary : exact Zs-DtN-CLN open boundary -- usage

[1] exact eddy/diffusion DtN per multipole (sphere R0=0.03 m):
    n=1: G_n(i*5e4) = -41.5082-40.4838j
    n=2: G_n(i*5e4) = -41.5329-40.4597j
    n=3: G_n(i*5e4) = -41.5699-40.4236j

[2] Cauer ladder is EXACT at n+1 stages and reproduces the symbol:
    n=1: stages=2 (= n+1)   ladder-vs-symbol NRMSE=1.6e-16
    n=2: stages=3 (= n+1)   ladder-vs-symbol NRMSE=1.4e-16
    n=3: stages=4 (= n+1)   ladder-vs-symbol NRMSE=1.7e-16
    n=4: stages=5 (= n+1)   ladder-vs-symbol NRMSE=1.6e-16
    n=5: stages=6 (= n+1)   ladder-vs-symbol NRMSE=1.3e-16
    n=6: stages=7 (= n+1)   ladder-vs-symbol NRMSE=1.4e-16

[3] transient Robin realisation: one auxiliary ODE per companion pole (Re<0):
    n=1: 1 poles, max Re = -1.000  => passive / unconditionally stable (Grote-Keller form)
    n=2: 2 poles, max Re = -1.500  => passive / unconditionally stable (Grote-Keller form)
    n=3: 3 poles, max Re = -1.839  => passive / unconditionally stable

    n=1: Kelvin-FEM build vs closed-form, band NRMSE=3.9e-03


    n=2: Kelvin-FEM build vs closed-form, band NRMSE=5.4e-04


    n=3: Kelvin-FEM build vs closed-form, band NRMSE=2.7e-04
    (the arbitrary-shape / iron-exterior path = ob.kelvin_dtn_matrix + ob.steklov_spectrum,
     verified O_h-split on a cube -- needs NGSolve; see tests/open_boundary/test_kelvin_dtn.py.
     NOTE: material-in-exterior Kelvin is CLASSICAL (Freeman-Lowther 1988/89).)

ALL CHECKS PASSED.


## Exact symbol does not mean a finite diffusion ODE

For spherical separation, the dimensionless diffusion DtN symbol used by
`eddy_dtn` is

$$G_n(s)=-q\frac{K_{n-1/2}(q)}{K_{n+1/2}(q)}-(n+1),
\qquad q=R\sqrt{s\mu\sigma}.$$

The physical normal derivative includes the corresponding $1/R$ scaling.
Exact boundary operators and their approximations are classical
[@keller1989exact; @hagstrom1999radiation]; Cauer reduction of eddy fields is
[@kameari2018cauer]. The reverse-Bessel identity is rational in **q**, not
generally rational in **s**. A finite exact ladder in q therefore does not
establish an exact finite-dimensional ODE in physical time for diffusion.
The square-root memory needs its own realization/approximation.

The saved assertions compare direct symbols, continued fractions and memory
approximations. Left-half-plane poles alone establish neither passivity nor
unconditional stability of an arbitrary timestep scheme; residues, port
signs and integrator also matter. No head-to-head PML timing/error campaign
is performed here, so a general claim of superiority over PML is not supported.
The Radia MCP open-boundary capability owns the current method selector.


## References

Generated from the canonical bibliography: [open_boundary_demo.bbl](open_boundary_demo.bbl).

<h3 class='likesectionHead' id='references'>  <a id='x1-1000'></a>References</h3>
<!-- l. 2 --><p class='noindent'>
   </p><section class='thebibliography' role='doc-bibliography'><dl><dt>
 [1]</dt><dd><a id='Xkeller1989exact'></a>J. B. Keller and D. Givoli, “Exact non-reflecting boundary conditions,” <span class='ecti-1000'>J.
   Comput. Phys.</span>, vol. 82, no. 1, pp. 172–192, 1989.
   </dd><dt>
 [2]</dt><dd><a id='Xhagstrom1999radiation'></a>T. Hagstrom,   “Radiation   Boundary   Conditions   for   the   Numerical
   Simulation of Waves,” <span class='ecti-1000'>Acta Numerica</span>, vol. 8, pp. 47–106, 1999.
   </dd><dt>
 [3]</dt><dd><a id='Xkameari2018cauer'></a>A. Kameari,  H. Ebrahimi,  K. Sugahara,  Y. Shindo,  and  T. Matsuo,
   “Cauer ladder network representation of eddy-current fields for model order
   reduction using finite-element method,” <span class='ecti-1000'>IEEE Transactions on Magnetics</span>,
   vol. 54, no. 3, p. 1–4, Mar 2018, art no. 7201804.
   </dd></dl></section>
